# Beyond Audio — Weighted Neural-Network Verification (NEW reconstruction)

**Purpose.** Check the class-weighted CNN/MM neural-network results reported in Appendix B by performing a new, controlled experiment using the documented architecture and training settings. Run the weighted and unweighted variants using the same fixed configurations and compare the resulting class predictions and metrics with the report.

**Provenance.** This notebook is a **new verification procedure**, prepared after the original experiments. It is **not the original weighted-training source**, and it does not prove that the historical weighted models were trained using exactly this procedure. No metrics are claimed to match until the notebook has been executed and the outputs inspected.

**Inputs.** The team's precomputed CNN/MM arrays, artist-disjoint label CSVs, and authorized Google Drive workspace; these are not distributed in the public GitHub repository. The upstream feature-extraction networks are not retrained.

**Outputs.** Only under `finetuning_experiments/reproducible_weighted_nn_2026/`, preserving the original model and prediction files. **Rerunning may overwrite files within this new output directory**; back up completed outputs before repeating a run.

**Suggested environment.** Google Colab with NVIDIA Tesla T4, close to the documented Stage 2 configuration. Exact matching across different PyTorch/CUDA hardware and software versions is not guaranteed.

## 1. Connect the data directory and record the environment
Use a new Google Colab runtime and enable a GPU (Runtime → Change runtime type). Confirm that the input paths and package versions match the documented environment before proceeding.

In [ ]:
from pathlib import Path
import os, sys, random, copy, json, platform
import numpy as np
import pandas as pd
import joblib
import sklearn
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, recall_score

SEED = 42
if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")

PROJECT = Path(os.environ.get("MSDI_PROJECT_DIR", "/content/drive/MyDrive/MSD-I_project"))
EXP = PROJECT / "finetuning_experiments"
CNN_DIR = EXP / "multimodal_artist_disjoint"
MM_DIR = EXP / "oramas_representation_experiment"
OUTPUT = EXP / "reproducible_weighted_nn_2026"  # New output folder only
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Project:", PROJECT)
print("New output directory:", OUTPUT)
print("Device:", DEVICE, torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
print("Python:", platform.python_version(), "NumPy:", np.__version__, "pandas:", pd.__version__)
print("scikit-learn:", sklearn.__version__, "PyTorch:", torch.__version__)
if not torch.cuda.is_available():
    print("WARNING: CPU training may not reproduce the original GPU-trained results.")

## 2. Verify dataset splits and feature alignment
The arrays are assumed to be in the same row order as the prepared label files. CNN and MM labels are checked for exact agreement on track IDs, artists, genres, and partitions. All model fitting uses training data only.

In [ ]:
EXPECTED_SIZES = {"train": 8242, "val": 1836, "test": 1792}
GENRES = ["Electronic", "Jazz", "Metal", "Pop", "Rap", "Rock"]
DIMS = {"CNN": (2048, 2048), "MM": (200, 200)}

labels = {}
raw = {}
for split in ("train", "val"):
    p_cnn = CNN_DIR / f"labels_{split}.csv"
    p_mm = MM_DIR / f"labels_{split}.csv"
    if not (p_cnn.exists() and p_mm.exists()):
        raise FileNotFoundError(f"Missing {split} metadata: {p_cnn}, {p_mm}")
    a, b = pd.read_csv(p_cnn), pd.read_csv(p_mm)
    for col in ("msd_track_id", "msd_artist_id", "genre", "split"):
        assert col in a.columns and col in b.columns, col
        assert a[col].astype(str).equals(b[col].astype(str)), (split, col)
    assert len(a) == EXPECTED_SIZES[split]
    assert a.msd_track_id.is_unique
    assert (a.split == split).all()
    labels[split] = a.reset_index(drop=True)

    for name in ("CNN", "MM"):
        if name == "CNN":
            a_path, v_path = CNN_DIR / f"X_{split}_audio.npy", CNN_DIR / f"X_{split}_visual.npy"
        else:
            a_path, v_path = MM_DIR / f"X_{split}_mm-audio.npy", MM_DIR / f"X_{split}_mm-visual.npy"
        A, V = np.load(a_path, allow_pickle=False), np.load(v_path, allow_pickle=False)
        assert A.shape == (EXPECTED_SIZES[split], DIMS[name][0]), (split, name, A.shape)
        assert V.shape == (EXPECTED_SIZES[split], DIMS[name][1]), (split, name, V.shape)
        assert np.isfinite(A).all() and np.isfinite(V).all()
        raw[name, split] = np.concatenate([A, V], axis=1).astype(np.float32)

assert set(labels["train"].msd_artist_id).isdisjoint(set(labels["val"].msd_artist_id))
assert set(labels["train"].msd_track_id).isdisjoint(set(labels["val"].msd_track_id))
encoder = LabelEncoder().fit(labels["train"].genre.astype(str))
assert encoder.classes_.tolist() == GENRES
Y = {s: encoder.transform(labels[s].genre.astype(str)).astype(np.int64) for s in ("train", "val")}

print("Verified train/validation sizes:", {s: len(labels[s]) for s in labels})
print("Artist disjoint:", set(labels["train"].msd_artist_id).isdisjoint(set(labels["val"].msd_artist_id)))

## 3. Train-only scaling and class weights
For genre `k`, the weights are `w_k = N_train/(6 N_k)`. This is the same inverse-frequency definition reported in Appendix B. PyTorch's weighted `CrossEntropyLoss` defaults to weighted-mean normalization over the mini-batch.

In [ ]:
scalers, prepared = {}, {}
for name in ("CNN", "MM"):
    scaler = StandardScaler()
    Xtr = scaler.fit_transform(raw[name, "train"]).astype(np.float32)
    Xval = scaler.transform(raw[name, "val"]).astype(np.float32)
    scalers[name] = scaler
    prepared[name] = {"train": Xtr, "val": Xval}
    assert np.isfinite(Xtr).all() and np.isfinite(Xval).all()

counts = np.bincount(Y["train"], minlength=len(GENRES))
assert (counts > 0).all()
expected_counts = np.array([1771, 918, 749, 1239, 868, 2697])
assert np.array_equal(counts, expected_counts), (counts, expected_counts)
class_weights = len(Y["train"]) / (len(GENRES) * counts.astype(float))
weights_table = pd.DataFrame({"genre": GENRES, "train_count": counts, "class_weight": class_weights})
print(weights_table.to_string(index=False, float_format=lambda x: f"{x:.6f}"))
assert np.allclose(class_weights * counts, class_weights[0] * counts[0])

## 4. Specify the architecture and model-selection protocol
Architectures and optimizer settings mirror the documented unweighted Stage 2 training. The **only intended experimental change** between a weighted and an unweighted run is whether the cross-entropy loss receives inverse-frequency class weights.

In [ ]:
class FusionMLP(nn.Module):
    def __init__(self, input_dim, hidden1, hidden2, dropout1, dropout2, num_classes=6):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden1), nn.ReLU(), nn.Dropout(dropout1),
            nn.Linear(hidden1, hidden2), nn.ReLU(), nn.Dropout(dropout2),
            nn.Linear(hidden2, num_classes),
        )
    def forward(self, x):
        return self.network(x)

CONFIGS = {
    "CNN": dict(input_dim=4096, hidden1=256, hidden2=64,
                dropout1=0.30, dropout2=0.20, lr=0.0005, weight_decay=0.01),
    "MM": dict(input_dim=400, hidden1=128, hidden2=32,
               dropout1=0.40, dropout2=0.30, lr=0.0003, weight_decay=0.05),
}

def build_model(name):
    cfg = CONFIGS[name]
    return FusionMLP(**{k: v for k, v in cfg.items() if k not in ("lr", "weight_decay")}).to(DEVICE)

for name in ("CNN", "MM"):
    model = build_model(name)
    n_params = sum(p.numel() for p in model.parameters())
    assert n_params == {"CNN": 1065670, "MM": 55654}[name]
    print(name, "trainable parameters:", n_params)
del model

## 5. Training helpers
All four runs reset seed 42 before model initialization and shuffled DataLoader iteration. Best checkpoint = highest validation macro-F1, requiring an improvement greater than `1e-4`; stop after 12 consecutive non-improving epochs, with a 100-epoch maximum.

**Important:** using the same seed does not guarantee identical runs on all hardware; it also does not establish that the missing historical weighted experiment used precisely this procedure.

In [ ]:
def reset_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def evaluate_predictions(model, X, batch_size=256):
    model.eval()
    out = []
    with torch.no_grad():
        for start in range(0, len(X), batch_size):
            xb = torch.from_numpy(X[start:start + batch_size]).to(DEVICE)
            out.append(model(xb).argmax(dim=1).cpu().numpy())
    return np.concatenate(out)


def train_variant(name, weighted):
    cfg = CONFIGS[name]
    tr_ds = TensorDataset(torch.from_numpy(prepared[name]["train"]), torch.from_numpy(Y["train"]))
    va_ds = TensorDataset(torch.from_numpy(prepared[name]["val"]), torch.from_numpy(Y["val"]))
    tr_loader = DataLoader(tr_ds, batch_size=128, shuffle=True, num_workers=0)
    va_loader = DataLoader(va_ds, batch_size=128, shuffle=False, num_workers=0)

    reset_seed(SEED)
    model = build_model(name)
    weight_tensor = torch.tensor(class_weights, dtype=torch.float32, device=DEVICE) if weighted else None
    loss_fn = nn.CrossEntropyLoss(weight=weight_tensor)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])
    best_f1, best_epoch, best_state = -float("inf"), 0, None
    stale, history = 0, []

    for epoch in range(1, 101):
        model.train()
        train_loss_sum = 0.0
        for xb, yb in tr_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            logits = model(xb)
            loss = loss_fn(logits, yb)
            loss.backward()
            opt.step()
            train_loss_sum += float(loss.item()) * len(yb)

        model.eval()
        val_preds, val_true = [], []
        with torch.no_grad():
            for xb, yb in va_loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                logits = model(xb)
                val_preds.extend(logits.argmax(dim=1).cpu().numpy())
                val_true.extend(yb.cpu().numpy())
        val_f1 = f1_score(val_true, val_preds, average="macro", zero_division=0)
        history.append(dict(epoch=epoch, train_loss=train_loss_sum / len(tr_ds), val_macro_f1=float(val_f1)))
        if val_f1 > best_f1 + 1e-4:
            best_f1, best_epoch = float(val_f1), epoch
            best_state = copy.deepcopy(model.state_dict())
            stale = 0
        else:
            stale += 1
        print(f"{name} {'Weighted' if weighted else 'Unweighted'} epoch {epoch:03d}: validation macro-F1={val_f1:.5f}")
        if stale >= 12:
            break

    assert best_state is not None
    model.load_state_dict(best_state)
    model.eval()
    val_pred = evaluate_predictions(model, prepared[name]["val"])
    assert abs(f1_score(Y["val"], val_pred, average="macro", zero_division=0) - best_f1) < 1e-12
    key = f"{name.lower()}_{'weighted' if weighted else 'unweighted'}"
    OUTPUT.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(history).to_csv(OUTPUT / f"{key}_training_history.csv", index=False)
    pd.DataFrame({"msd_track_id": labels["val"].msd_track_id,
                  "genre": labels["val"].genre,
                  "prediction": encoder.inverse_transform(val_pred)}).to_csv(
                      OUTPUT / f"{key}_validation_predictions.csv", index=False)
    torch.save({"model_state_dict": best_state, "best_epoch": best_epoch,
                "best_val_macro_f1": best_f1, "weighted": weighted,
                "class_weights": class_weights.tolist() if weighted else None,
                "architecture": cfg, "seed": SEED}, OUTPUT / f"{key}_best.pt")
    joblib.dump(scalers[name], OUTPUT / f"{key}_scaler.joblib")
    return {"model": model, "val_macro_f1": best_f1, "best_epoch": best_epoch,
            "weighted": weighted, "name": name}

## 6. Run the controlled CNN experiments
First run the unweighted CNN reconstruction, then its weighted counterpart. The unweighted run is a reproduction control: if it fails to match the verified historical outputs under the documented environment, interpret subsequent weighted results cautiously.

In [ ]:
RUNS = {}
RUNS[("CNN", "U")] = train_variant("CNN", weighted=False)
RUNS[("CNN", "W")] = train_variant("CNN", weighted=True)
print("CNN selected epochs:", {k: v["best_epoch"] for k,v in RUNS.items()})

## 7. Run the controlled MM experiments

In [ ]:
RUNS[("MM", "U")] = train_variant("MM", weighted=False)
RUNS[("MM", "W")] = train_variant("MM", weighted=True)
print("All selected epochs:", {str(k): v["best_epoch"] for k,v in RUNS.items()})

## 8. Evaluate once on the held-out test set
The test labels and test features enter only **after all four training runs and validation checkpoint selections are complete**. No hyperparameters or epochs are adjusted based on test performance.

In [ ]:
cnn_test_labels = pd.read_csv(CNN_DIR / "labels_test.csv")
mm_test_labels = pd.read_csv(MM_DIR / "labels_test.csv")
for col in ("msd_track_id", "msd_artist_id", "genre", "split"):
    assert cnn_test_labels[col].astype(str).equals(mm_test_labels[col].astype(str)), col
assert len(cnn_test_labels) == EXPECTED_SIZES["test"]
assert cnn_test_labels.msd_track_id.is_unique
assert (cnn_test_labels.split == "test").all()
for split in ("train", "val"):
    assert set(cnn_test_labels.msd_track_id).isdisjoint(set(labels[split].msd_track_id))
    assert set(cnn_test_labels.msd_artist_id).isdisjoint(set(labels[split].msd_artist_id))

true_test = encoder.transform(cnn_test_labels.genre.astype(str))
X_test = {}
for name in ("CNN", "MM"):
    if name == "CNN":
        a, v = CNN_DIR / "X_test_audio.npy", CNN_DIR / "X_test_visual.npy"
    else:
        a, v = MM_DIR / "X_test_mm-audio.npy", MM_DIR / "X_test_mm-visual.npy"
    A, V = np.load(a, allow_pickle=False), np.load(v, allow_pickle=False)
    assert A.shape == (1792, DIMS[name][0]) and V.shape == (1792, DIMS[name][1])
    combined = np.concatenate([A, V], axis=1).astype(np.float32)
    assert np.isfinite(combined).all()
    X_test[name] = scalers[name].transform(combined).astype(np.float32)

summary_rows, recall_rows = [], []
predictions = cnn_test_labels[["msd_track_id", "msd_artist_id", "genre"]].copy()
for name in ("CNN", "MM"):
    for suffix in ("U", "W"):
        run = RUNS[(name, suffix)]
        p = evaluate_predictions(run["model"], X_test[name])
        predictions[f"{name.lower()}_{suffix.lower()}_prediction"] = encoder.inverse_transform(p)
        summary_rows.append({
            "model": f"{name} {suffix}",
            "best_epoch": run["best_epoch"],
            "val_macro_f1": 100 * run["val_macro_f1"],
            "accuracy": 100 * accuracy_score(true_test, p),
            "balanced_accuracy": 100 * balanced_accuracy_score(true_test, p),
            "macro_f1": 100 * f1_score(true_test, p, average="macro", zero_division=0),
        })
        recalls = 100 * recall_score(true_test, p, labels=np.arange(len(GENRES)), average=None, zero_division=0)
        recall_rows.append({"model": f"{name} {suffix}", **dict(zip(GENRES, recalls))})

summary = pd.DataFrame(summary_rows)
recalls = pd.DataFrame(recall_rows).set_index("model")
print("Experimental results (%) — NEW runs, not yet validated against historical weights:")
print(summary.to_string(index=False, float_format=lambda x: f"{x:.2f}"))
print("\nPer-genre test recall (%):")
print(recalls.to_string(float_format=lambda x: f"{x:.2f}"))

### Optional control — compare new *unweighted* predictions with documented Stage 2 output
If the Stage 2 verification outputs are available in Google Drive, compare track-level validation and test predictions. Exact agreement is stronger evidence that the present configuration reproduces the known unweighted baseline. **This comparison never alters training or selects new checkpoints.**

In [ ]:
stage2_dir = EXP / "reproducible_stage2_2026"
for name in ("CNN", "MM"):
    old_val_path = stage2_dir / f"{name.lower()}_fusion_mlp_validation_predictions.csv"
    new_val_path = OUTPUT / f"{name.lower()}_unweighted_validation_predictions.csv"
    if old_val_path.exists():
        old = pd.read_csv(old_val_path)[["msd_track_id", "prediction"]].rename(columns={"prediction": "old"})
        new = pd.read_csv(new_val_path)[["msd_track_id", "prediction"]].rename(columns={"prediction": "new"})
        merged = old.merge(new, on="msd_track_id", validate="one_to_one")
        assert len(merged) == EXPECTED_SIZES["val"]
        print(name, "validation differences vs Stage 2:", int((merged.old != merged.new).sum()))
    else:
        print(name, "Stage 2 validation predictions unavailable — skipped")

old_test_path = stage2_dir / "stage2_test_predictions.csv"
if old_test_path.exists():
    old = pd.read_csv(old_test_path)
    for name in ("CNN", "MM"):
        historical_col = f"{name.lower()}_nn_prediction"
        new_col = f"{name.lower()}_u_prediction"
        if historical_col not in old.columns:
            print(name, "Stage 2 test prediction column unavailable — skipped")
            continue
        merged = old[["msd_track_id", historical_col]].merge(
            predictions[["msd_track_id", new_col]], on="msd_track_id", validate="one_to_one")
        assert len(merged) == EXPECTED_SIZES["test"]
        print(name, "test differences vs Stage 2:", int((merged[historical_col] != merged[new_col]).sum()))
else:
    print("Stage 2 test predictions unavailable — skipped")

## 9. Compare with the report without silently altering the targets
The values below are **reference values from Appendix B**, not results generated by this notebook. A mismatch is an outcome to document and investigate, **not** a reason to tune on the test set. Exact agreement is environment-sensitive, especially for neural-network training.

In [ ]:
REPORT = pd.DataFrame([
    ["CNN U", 67.38, 68.58, 67.90, 67.97],
    ["CNN W", 67.09, 68.25, 70.24, 68.23],
    ["MM U", 68.85, 69.03, 68.31, 68.49],
    ["MM W", 68.14, 68.02, 70.93, 68.17],
], columns=["model", "val_macro_f1", "accuracy", "balanced_accuracy", "macro_f1"])
comparison = summary.merge(REPORT, on="model", validate="one_to_one", suffixes=("_new", "_report"))
metrics = ["val_macro_f1", "accuracy", "balanced_accuracy", "macro_f1"]
for m in metrics:
    comparison[f"{m}_delta_pp"] = comparison[f"{m}_new"] - comparison[f"{m}_report"]
comparison["max_abs_difference_pp"] = comparison[[f"{m}_delta_pp" for m in metrics]].abs().max(axis=1)
print(comparison[["model", "best_epoch", "max_abs_difference_pp"] + [f"{m}_delta_pp" for m in metrics]].to_string(index=False, float_format=lambda x: f"{x:+.3f}"))
print("\nNOTE: deviations do not invalidate historical values; they indicate what this NEW verification run did or did not reproduce.")

RECALL_REPORT = pd.DataFrame({
    "CNN U": [75.40, 61.57, 66.23, 53.67, 79.77, 70.76],
    "CNN W": [71.43, 68.12, 76.16, 55.21, 85.55, 64.95],
    "MM U": [72.75, 57.21, 70.86, 51.74, 83.24, 74.09],
    "MM W": [70.37, 65.50, 80.13, 59.07, 87.86, 62.62],
}, index=GENRES).T
recall_difference = recalls - RECALL_REPORT.loc[recalls.index, GENRES]
print("\nLargest absolute genre recall difference by model (percentage points):")
print(recall_difference.abs().max(axis=1).to_string(float_format=lambda x: f"{x:.3f}"))

## 10. Save evidence, provenance, and a concise verification summary
This notebook cannot certify the original weighted training history; record execution date, software, device, model-selection epochs, observed differences, and any issues. **Do not edit the report's stated metrics based on new test runs until the discrepancy has been understood and a valid evaluation policy established.**

In [ ]:
OUTPUT.mkdir(parents=True, exist_ok=True)
summary.to_csv(OUTPUT / "new_weighted_comparison_metrics.csv", index=False)
recalls.to_csv(OUTPUT / "new_weighted_comparison_genre_recall.csv")
comparison.to_csv(OUTPUT / "new_weighted_vs_report_metrics.csv", index=False)
recall_difference.to_csv(OUTPUT / "new_weighted_vs_report_recall_deltas.csv")
predictions.to_csv(OUTPUT / "new_weighted_comparison_test_predictions.csv", index=False)
weights_table.to_csv(OUTPUT / "new_weighted_class_weights.csv", index=False)

record = {
    "purpose": "NEW reconstruction verifying previously reported weighted NN results",
    "is_original_historical_weighted_training": False,
    "raw_data_to_paper_pipeline": False,
    "seed": SEED,
    "python": platform.python_version(),
    "torch": torch.__version__,
    "sklearn": sklearn.__version__,
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "device": str(DEVICE),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "train_recordings": int(len(Y["train"])),
    "validation_recordings": int(len(Y["val"])),
    "test_recordings": int(len(true_test)),
    "trained_variants": list(summary["model"]),
    "best_epochs": {row["model"]: int(row["best_epoch"]) for _, row in summary.iterrows()},
    "warning": "This checks a reconstructed procedure; it does not establish historical weighted-training provenance.",
}
with open(OUTPUT / "new_weighted_verification_metadata.json", "w", encoding="utf-8") as f:
    json.dump(record, f, indent=2)
print("Saved ONLY to", OUTPUT)
print("Next: inspect the results, then document what matched and what differed.")